In [46]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Environment stable.")

Environment stable.


## **1. Load the JCI ESS document**

In [47]:
JCI_PDF = "/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf"

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(JCI_PDF)
pages = loader.load()

print(f"Total pages loaded = {len(pages)}")


Total pages loaded = 406


## **2. Grouping pages by chapter**

In [50]:
import re
from collections import defaultdict


VALID_CODES = {
    "APR","IPSG","ACC","PCC","AOP","COP","ASC","MMU",
    "QPS","PCI","GLD","FMS","SQE","MOI",
    "MPE","HRP"
}

def detect_chapter_from_text(text: str):
    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        match = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line)
        if match:
            code = match.group(1)               
            if code in VALID_CODES:
                return code

    return None


chapter_pages = defaultdict(list)
current_chapter = "UNKNOWN"
#Search logic for page with that particular heading's content: check if page
for page in pages:
    detected = detect_chapter_from_text(page.page_content)

    if detected:
        current_chapter = detected

    chapter_pages[current_chapter].append(page)

# Print result
total = 0
for chapter, pg in chapter_pages.items():
    print(f"{chapter}: {len(pg)} pages")
    total += len(pg)

print("Total grouped pages:", total)

UNKNOWN: 2 pages
FMS: 42 pages
APR: 18 pages
ACC: 22 pages
PCC: 21 pages
AOP: 30 pages
COP: 32 pages
ASC: 16 pages
QPS: 15 pages
PCI: 27 pages
GLD: 32 pages
SQE: 28 pages
MOI: 25 pages
MPE: 13 pages
HRP: 45 pages
IPSG: 16 pages
MMU: 22 pages
Total grouped pages: 406


In [51]:
import re

acc_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])

matches = re.findall(r'[A-Z]{3,4}\.\d+(?:\.\d+)?', acc_text)

print("Detected standards:", matches[:20])
print("Total detected:", len(matches))

Detected standards: ['IPSG.6', 'IPSG.6.1', 'ACC.1', 'ACC.1.1', 'ACC.1.1', 'ACC.1.1', 'ACC.1.1', 'ACC.1.2', 'ACC.2', 'ACC.2.2', 'ACC.2.2', 'ACC.2.2', 'ACC.2.2', 'ACC.2.3', 'ACC.2.3', 'ACC.3', 'ACC.3.2', 'ACC.3.2', 'ACC.3.2', 'ACC.3.1']
Total detected: 155


In [75]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_chapter_chunks(chapter_pages, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        docs = splitter.split_documents(pg_list)
        chapter_chunks[chapter] = docs
        print(f"{chapter}: {len(docs)} chunks")

    return chapter_chunks


def build_structured_chapter_chunks(chapter_pages, chunk_size=800, chunk_overlap=150):
    
    inner_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "]
    )

    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        
        # 1. Merge all pages into one string
        full_text = "\n".join([p.page_content for p in pg_list])

        # 2. Clean page artifacts
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        # 3. Skip the TOC/summary section at the top
        # The real content starts at "Standards, Intents, and Measurable Elements"
        # which appears as a heading before the first real standard
        toc_marker = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
            full_text,
            re.IGNORECASE
        )
        if toc_marker:
            full_text = full_text[toc_marker.start():]
        
        # 4. Split by standard codes that BELONG to this chapter only
        pattern = rf'(?:\n|^)Standard\s+({chapter}\.\d+(?:\.\d+)?)\b'
        splits = re.split(pattern, full_text)

        docs = []

        # splits[0] is intro text before any standard code — keep it
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={"chapter": chapter, "standard_code": "INTRO"}
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break

            standard_code = splits[i].strip()
            content = f"{standard_code}\n{splits[i+1]}"

            if len(content.strip()) < 100:
                continue

            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={
                        "chapter": chapter,
                        "standard_code": standard_code
                    }
                ))
            else:
                sub_chunks = inner_splitter.split_text(content)
                for j, chunk_text in enumerate(sub_chunks):
                    if len(chunk_text.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=chunk_text.strip(),
                        metadata={
                            "chapter": chapter,
                            "standard_code": standard_code,
                            "chunk_index": j
                        }
                    ))

        chapter_chunks[chapter] = docs
        print(f"{chapter}: {len(docs)} chunks across "
              f"{len(set(d.metadata['standard_code'] for d in docs))} standards")

    return chapter_chunks


# chapter_chunks = build_chapter_chunks(
#     chapter_pages,
#     chunk_size=1000,
#     chunk_overlap=200
# )

chapter_chunks = build_structured_chapter_chunks(
    chapter_pages,
    chunk_size=1000,
    chunk_overlap=250
)

UNKNOWN: 1 chunks across 1 standards
FMS: 131 chunks across 26 standards
APR: 1 chunks across 1 standards
ACC: 80 chunks across 17 standards
PCC: 79 chunks across 22 standards
AOP: 1 chunks across 1 standards
COP: 1 chunks across 1 standards
ASC: 50 chunks across 16 standards
QPS: 56 chunks across 12 standards
PCI: 106 chunks across 22 standards
GLD: 142 chunks across 34 standards
SQE: 128 chunks across 25 standards
MOI: 95 chunks across 17 standards
MPE: 23 chunks across 8 standards
HRP: 1 chunks across 1 standards
IPSG: 76 chunks across 14 standards
MMU: 85 chunks across 18 standards


In [76]:
# DIAGNOSTIC 2: Standard code splitting
print("=== STANDARD CODE SPLITTING RESULTS ===")

# Pick ACC as our test chapter since we know it has 22 pages
test_chapter = "ACC"
test_docs = chapter_chunks[test_chapter]

print(f"\nTotal chunks in {test_chapter}: {len(test_docs)}")

# What unique standard codes were detected?
unique_codes = sorted(set(doc.metadata["standard_code"] for doc in test_docs))
print(f"\nUnique standard codes found ({len(unique_codes)}):")
for code in unique_codes:
    # How many chunks per code?
    count = sum(1 for doc in test_docs if doc.metadata["standard_code"] == code)
    print(f"  {code}: {count} chunks")

# Check for any suspiciously short chunks (likely bad splits)
short_chunks = [doc for doc in test_docs if len(doc.page_content) < 100]
print(f"\nSuspiciously short chunks (< 100 chars): {len(short_chunks)}")
if short_chunks:
    print("Example short chunk:")
    print(repr(short_chunks[0].page_content))

=== STANDARD CODE SPLITTING RESULTS ===

Total chunks in ACC: 80

Unique standard codes found (17):
  ACC.1: 6 chunks
  ACC.1.1: 2 chunks
  ACC.2: 6 chunks
  ACC.2.1: 1 chunks
  ACC.2.2: 2 chunks
  ACC.2.3: 5 chunks
  ACC.3: 6 chunks
  ACC.3.1: 3 chunks
  ACC.4: 1 chunks
  ACC.4.1: 8 chunks
  ACC.4.2: 5 chunks
  ACC.4.3: 3 chunks
  ACC.4.4: 7 chunks
  ACC.5: 6 chunks
  ACC.5.1: 3 chunks
  ACC.6: 15 chunks
  INTRO: 1 chunks

Suspiciously short chunks (< 100 chars): 0


In [77]:
# DIAGNOSTIC 3: Metadata check
print("=== METADATA CHECK ===")

sample_docs = chapter_chunks["ACC"]

# Check every chunk has required metadata fields
missing_chapter = [d for d in sample_docs if "chapter" not in d.metadata]
missing_code = [d for d in sample_docs if "standard_code" not in d.metadata]

print(f"Chunks missing 'chapter': {len(missing_chapter)}")
print(f"Chunks missing 'standard_code': {len(missing_code)}")

# Show metadata from 3 different standards to confirm it looks right
for code in ["ACC.1", "ACC.3", "ACC.6"]:
    example = next((d for d in sample_docs if d.metadata["standard_code"] == code), None)
    if example:
        print(f"\n--- Example chunk from {code} ---")
        print("Metadata:", example.metadata)
        print("Text preview:", example.page_content[:200])

=== METADATA CHECK ===
Chunks missing 'chapter': 0
Chunks missing 'standard_code': 0

--- Example chunk from ACC.1 ---
Metadata: {'chapter': 'ACC', 'standard_code': 'ACC.1', 'chunk_index': 1}
Text preview: Patients who may be admitted to the hospital or who seek outpatient services are screened to identify if their 
health care needs match the hospital’s mission and resources, and those with emergent, u

--- Example chunk from ACC.3 ---
Metadata: {'chapter': 'ACC', 'standard_code': 'ACC.3', 'chunk_index': 1}
Text preview: The hospital designs and carries out processes to provide continuity of patient care services in the hospital, 
coordination among health care practitioners, and access to information related to the p

--- Example chunk from ACC.6 ---
Metadata: {'chapter': 'ACC', 'standard_code': 'ACC.6', 'chunk_index': 1}
Text preview: The hospital’s transportation services comply with relevant laws and regulations and meet requirements for 
quality and safe transport. 𝖯
Intent of AC

In [71]:
# DIAGNOSTIC 3: Investigate ACC.6
acc6_chunks = [doc for doc in chapter_chunks["ACC"] if doc.metadata["standard_code"] == "ACC.6"]

print(f"ACC.6 total chunks: {len(acc6_chunks)}")
print(f"ACC.6 total characters: {sum(len(d.page_content) for d in acc6_chunks)}")

print("\n--- First chunk ---")
print(acc6_chunks[0].page_content[:500])

print("\n--- Last chunk ---")
print(acc6_chunks[-1].page_content[:500])

ACC.6 total chunks: 76
ACC.6 total characters: 72788

--- First chunk ---
ACC.6
 The hospital’s transportation services comply with relevant laws and regulations and meet 
requirements for quality and safe transport. 𝖯
Access to Care and 
Continuity of Care (ACC)
41
aCCeSS to Care and ContInuIty of Care (aCC)
Standards, Intents, and Measurable Elements
Screening for Admission to the Hospital
Standard ACC.1
Patients who may be admitted to the hospital or who seek outpatient services are screened to identify if their 
health care needs match the hospital’s mission and r

--- Last chunk ---
Williams P , et al. A checklist for intrahospital transport of 
critically ill patients improves compliance with transportation 
safety guidelines. Aust Crit Care. Epub 2019 Apr 10. https://
doi.org/10.1016/j.aucc.2019.02.004.
Transportation
Overton JW . Safety and Quality in Medical Transport Systems: 
Creating an Effective Culture. Boca Raton, FL: CRC Press, 
2012. https://doi.org/10.1201/97813156074

In [72]:
# DIAGNOSTIC 4: Check what's at the start of ACC chapter
acc_full_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])
print(acc_full_text[:2000])

xi
Summary of ChangeS to the manual
• Intent of IPSG.6 and IPSG.6.1: Added information about high-risk areas for falls and fall risk 
screening
• Made minor editorial revisions throughout the chapter
Access to Care and Continuity of Care (ACC)
• ACC.1
o Standard and Intent: Combined with previous Standard ACC.1.1 and revised to include 
requirements on prioritization of patients with emergent, urgent, or immediate needs
o ME 1: Combined with previous MEs 2 and 6; clarified that patients are accepted for treatment if 
it is determined that the needs of the patient match the hospital’s mission and resources
o ME 2: Combined previous ME 2 with MEs 1 and 6 and numbered as ME 1
o ME 3: Renumbered previous ACC.1.1, ME 1
o ME 4: Combined and renumbered requirements of previous ACC.1.1, MEs 4 and 5 
o ME 5: Renumbered previous ME 4
• ACC.1.1
o Standard and MEs: Renumbered previous Standard ACC.1.2 and its MEs 1–5
o Intent: Added examples of situations that would require the patient to be notif

In [73]:
# DIAGNOSTIC 5: Find exactly where ACC.6 appears in the raw text
acc_full_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])

# Find ALL occurrences of ACC.6 in the text
for match in re.finditer(r'ACC\.6', acc_full_text):
    start = max(0, match.start() - 200)
    end = min(len(acc_full_text), match.end() + 200)
    print(f"\n--- ACC.6 found at position {match.start()} ---")
    print(repr(acc_full_text[start:end]))
    print()


--- ACC.6 found at position 4797 ---
'health care organizations\no Intent: Combined with intent of previous Standard ACC.5.1\no ME 2: Expanded to address how and when responsibility for a patient is transferred \no ME 3: Renumbered previous ACC.6, ME 3 and added new requirement to identify an \nindividual responsible for the patient during transfer \no MEs 4–6: Renumbered previous ACC.5.1, MEs 3–5\n• ACC.5.1\no Standard: Combined and renumbered p'


--- ACC.6 found at position 5265 ---
'umbered previous ACC.5.2, ME 1 and ACC.5.3, MEs 2–4\no ME 2: Renumbered previous ACC.5.3, ME 1\no ME 3: Renumbered previous ACC.5.3, ME 2\no ME 4: Combined and renumbered previous ACC.5.3, MEs 3 and 4\n• ACC.6\no Intent: Expanded discussion of patient transportation needs and added examples\n39\nAccess to Care and \nContinuity of Care \n(ACC)\nOverview\nHealth care organizations are pursuing a more comprehensive '


--- ACC.6 found at position 10626 ---
's given a written summary of the patient’s clini

In [74]:
# DIAGNOSTIC 6: Find what text appears just before position 56620
acc_full_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])

# What's around position 56000-56700?
print(repr(acc_full_text[55800:56700]))

'icy (for example, a signature of the receiving nurse or physician, the \nname of the individual who monitored the patient during transport) is included in the medical record. (Also see \nMOI.8)\nMeasurable Elements of ACC.5.1\nq 1. A patient clinical summary document is transferred with the patient and includes at least a) through \nc) of the intent.\nq 2. The medical records of transferred patients identify the name of the receiving health care organization \nand the name of the individual agreeing to receive the patient.\nq 3. The medical records of transferred patients contain documentation or other notes as required by the \npolicy of the transferring hospital.\nq 4. The medical records of transferred patients state the reason(s) for transfer and any special conditions \nrelated to transfer.\nTransportation\nStandard ACC.6\nThe hospital’s transportation services comply with relevant laws and regul'


In [ ]:
for doc in chapter_chunks["ACC"][:2]:
    print(doc.metadata)
    print(doc.page_content[:300])
    print("-----")

{'chapter': 'ACC', 'standard_code': 'INTRO'}
Standards, Intents, and Measurable Elements.
Screening for Admission to the Hospital
ACC.1 Patients who may be admitted to the hospital or who seek outpatient services are screened to 
identify if their health care needs match the hospital’s mission and resources, and those with 
emergent, urgent, o
-----
{'chapter': 'ACC', 'standard_code': 'ACC.1', 'chunk_index': 1}
Patients who may be admitted to the hospital or who seek outpatient services are screened to identify if their 
health care needs match the hospital’s mission and resources, and those with emergent, urgent, or immediate 
needs are given priority for assessment and treatment. 𝖯
Intent of ACC.1
Matchi
-----


In [13]:
#inspecting 2 example chunks from ACC

for i, doc in enumerate(chapter_chunks["ACC"][:2]):
    print(f"\n--- Chunk {i+1} ---")
    print("Metadata:", doc.metadata)
    print("Text preview:\n")
    print(doc.page_content[:500])
    print("\n-----")


--- Chunk 1 ---
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '2022-07-07T16:18:36-05:00', 'moddate': '2023-10-20T11:40:30+08:00', 'trapped': '/False', 'source': '/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf', 'total_pages': 406, 'page': 10, 'page_label': 'xi'}
Text preview:

xi
Summary of ChangeS to the manual
• Intent of IPSG.6 and IPSG.6.1: Added information about high-risk areas for falls and fall risk 
screening
• Made minor editorial revisions throughout the chapter
Access to Care and Continuity of Care (ACC)
• ACC.1
o Standard and Intent: Combined with previous Standard ACC.1.1 and revised to include 
requirements on prioritization of patients with emergent, urgent, or immediate needs
o ME 1: Combined with previous MEs 2 and 6; clarified that patients are acce

-----

--- Chunk 2 ---
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Windows)', 'creationdate': '20

## **3. Embedding model - local/open source**

In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1046.43it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


## **4. Builing vector DB for each chapter**

In [16]:
jci_dbs = {}

for chapter, docs in chapter_chunks.items():
    if docs:
        print(f"Building FAISS for {chapter}...")
        jci_dbs[chapter] = FAISS.from_documents(docs, embeddings)

print("Built DBs for:", list(jci_dbs.keys()))

Building FAISS for UNKNOWN...
Building FAISS for FMS...
Building FAISS for APR...
Building FAISS for ACC...
Building FAISS for PCC...
Building FAISS for AOP...
Building FAISS for COP...
Building FAISS for ASC...
Building FAISS for QPS...
Building FAISS for PCI...
Building FAISS for GLD...
Building FAISS for SQE...
Building FAISS for MOI...
Building FAISS for MPE...
Building FAISS for HRP...
Building FAISS for IPSG...
Building FAISS for MMU...
Built DBs for: ['UNKNOWN', 'FMS', 'APR', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'QPS', 'PCI', 'GLD', 'SQE', 'MOI', 'MPE', 'HRP', 'IPSG', 'MMU']


In [17]:
#Save indices per chapter
import os

BASE_SAVE_DIR = "faiss_jci_chapters"
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

for chapter, db in jci_dbs.items():
    save_path = os.path.join(BASE_SAVE_DIR, chapter)
    db.save_local(save_path)

print("All chapter FAISS indices saved.")

All chapter FAISS indices saved.


In [18]:
#testing retrieval
test_query = "patient transfer communication"

results = jci_dbs["ACC"].similarity_search(test_query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}")
    print(r.page_content[:400])
    print("-----")


Result 1
significantly different. Thus, the condition and status of the patient determine the qualifications of the staff 
member monitoring the patient and the type of medical equipment needed during transfer.
A consistent process for how patients are transferred from one organization to another is required to ensure 
that patients are transferred safely. Such a process addresses
• how and when responsibi
-----

Result 2
of the patient’s care, it needs to be available during inpatient care, for outpatient visits, and at other times as 
needed and kept up to date. Medical, nursing, and other patient care notes are available to all of the patient’s 
health care practitioners who need them for the care of the patient. (Also see MOI.5)
When the care team changes as a result of a transfer, continuity of patient care re
-----

Result 3
staff qualifications required for the type of patient being transferred.
q 4. The transfer process identifies the medications, supplies, and medical equipme

In [78]:
# DIAGNOSTIC 4: Retrieval quality check
print("=== RETRIEVAL QUALITY CHECK ===")

# Rebuild the ACC FAISS db with the new chunks (important - old db used old chunks)
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
jci_dbs["ACC"] = FAISS.from_documents(chapter_chunks["ACC"], embeddings)

# Test 3 queries that map to known standards
test_queries = [
    ("patient transfer communication handover", "ACC.3 or ACC.5"),
    ("transportation safety laws regulations",  "ACC.6"),
    ("screening admission emergency urgent",    "ACC.1"),
]

for query, expected in test_queries:
    print(f"\nQuery: '{query}'")
    print(f"Expected standard: {expected}")
    results = jci_dbs["ACC"].similarity_search(query, k=3)
    for i, r in enumerate(results):
        print(f"  Result {i+1}: {r.metadata['standard_code']} | {r.page_content[:100]}")

=== RETRIEVAL QUALITY CHECK ===


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1269.53it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: 'patient transfer communication handover'
Expected standard: ACC.3 or ACC.5
  Result 1: ACC.5 | qualified staff and the correct medical equipment for the patient’s condition.
Measurable Elements o
  Result 2: ACC.5 | that patients are transferred safely. Such a process addresses
• how and when responsibility is tran
  Result 3: ACC.5 | T ransferring a patient directly to another health care organization may be a brief process with an 

Query: 'transportation safety laws regulations'
Expected standard: ACC.6
  Result 1: ACC.6 | required drugs, medications, and other supplies needed within the vehicle are based on the types of 
  Result 2: ACC.6 | quality decisions related to patient transports.
In all cases, the hospital evaluates the quality an
  Result 3: ACC.6 | incident study. Aust Crit Care. Epub 2019 Jan 21. doi: https://
doi.org/10.1016/j.aucc.2018.12.003.


Query: 'screening admission emergency urgent'
Expected standard: ACC.1
  Result 1: ACC.1 | Patients who may be admi

# 7 Gold Standards

In [79]:
from collections import Counter

gold_standard = [
    # ACC.1 — Screening for admission
    {"query": "How should hospitals evaluate whether a patient's needs match available services?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
    {"query": "What are the requirements for triaging patients with urgent or emergency needs?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
    # ACC.1.1 — Delays in service
    {"query": "What must hospitals do when there are delays in providing care to patients?", "expected_standard": "ACC.1.1", "expected_chapter": "ACC"},
    # ACC.2 — Patient flow
    {"query": "How should hospitals manage patient flow and bed allocation throughout the facility?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},
    # ACC.2.3 — Admission and discharge criteria
    {"query": "What criteria should be used to admit or discharge patients from specialised units?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    # ACC.3 — Continuity of care
    {"query": "How should information be shared between healthcare practitioners to ensure continuity?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
    {"query": "What processes ensure care coordination when a patient moves between departments?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
    # ACC.4 — Discharge planning
    {"query": "When should discharge planning begin and what should it include?", "expected_standard": "ACC.4", "expected_chapter": "ACC"},
    # ACC.4.1 — Patient education on discharge
    {"query": "What instructions and information must be given to patients when they leave the hospital?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},
    # ACC.5 — Transfer
    {"query": "What are the requirements for safely transferring a patient to another healthcare facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
    {"query": "Who is responsible for monitoring a patient during transfer and what qualifications are needed?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
    # ACC.5.1 — Transfer documentation
    {"query": "What clinical information must be documented and handed over when transferring a patient?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},
    # ACC.6 — Transportation
    {"query": "What safety and regulatory standards must hospital transportation services meet?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
    {"query": "How should hospitals assess and arrange safe transport for patients being discharged?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
    {"query": "What equipment and medications must be available in patient transport vehicles?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
]

print(f"Gold standard: {len(gold_standard)} queries")
standard_counts = Counter(item["expected_standard"] for item in gold_standard)
print("\nQueries per standard:")
for standard, count in sorted(standard_counts.items()):
    print(f"  {standard}: {count} queries")

Gold standard: 15 queries

Queries per standard:
  ACC.1: 2 queries
  ACC.1.1: 1 queries
  ACC.2: 1 queries
  ACC.2.3: 1 queries
  ACC.3: 2 queries
  ACC.4: 1 queries
  ACC.4.1: 1 queries
  ACC.5: 2 queries
  ACC.5.1: 1 queries
  ACC.6: 3 queries


# 8 Model definitions

In [80]:
# Models to compare
# We load one at a time during evaluation to avoid memory issues
MODELS_TO_EVALUATE = {
    "MiniLM (baseline)" : "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small"         : "BAAI/bge-small-en-v1.5",
    "BGE-large"         : "BAAI/bge-large-en-v1.5",
}

# k values to test
# hit@3 = did correct standard appear in top 3 results?
K_VALUES = [1, 3, 5]

print("Models to evaluate:")
for name, model_str in MODELS_TO_EVALUATE.items():
    print(f"  {name}: {model_str}")

Models to evaluate:
  MiniLM (baseline): sentence-transformers/all-MiniLM-L6-v2
  BGE-small: BAAI/bge-small-en-v1.5
  BGE-large: BAAI/bge-large-en-v1.5


# 9 Evaluation functions

In [81]:
import time

def run_evaluation(chapter_chunks, gold_standard, model_name, model_string, k_values):
    print(f"\n{'='*50}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*50}")

    # Load embedding model
    print("Loading model...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_string,
        encode_kwargs={"normalize_embeddings": True}
    )

    # Build FAISS DBs only for chapters we need
    needed_chapters = set(item["expected_chapter"] for item in gold_standard)
    print(f"Building DBs for: {needed_chapters}")
    dbs = {}
    for chapter in needed_chapters:
        if chapter in chapter_chunks and chapter_chunks[chapter]:
            dbs[chapter] = FAISS.from_documents(chapter_chunks[chapter], embeddings)
            print(f"  Built {chapter} DB")

    # Run every query
    results = []
    max_k = max(k_values)

    for item in gold_standard:
        query             = item["query"]
        expected_standard = item["expected_standard"]
        chapter           = item["expected_chapter"]

        db = dbs.get(chapter)
        if db is None:
            continue

        retrieved       = db.similarity_search(query, k=max_k)
        retrieved_codes = [r.metadata["standard_code"] for r in retrieved]

        row = {
            "model":             model_name,
            "query":             query,
            "expected_standard": expected_standard,
            "retrieved_codes":   retrieved_codes,
        }
        # Check if expected standard appears in top k results
        for k in k_values:
            row[f"hit@{k}"] = expected_standard in retrieved_codes[:k]

        results.append(row)

    # Print summary
    print(f"\nResults for {model_name}:")
    for k in k_values:
        hits     = sum(1 for r in results if r[f"hit@{k}"])
        hit_rate = hits / len(results) * 100
        print(f"  Hit Rate @{k}: {hit_rate:.0f}% ({hits}/{len(results)})")

    return results

print("Evaluation function defined.")

Evaluation function defined.


# 10 Running evalutation

In [83]:
import pandas as pd

all_results = []

for model_name, model_string in MODELS_TO_EVALUATE.items():
    start   = time.time()
    results = run_evaluation(
        chapter_chunks=chapter_chunks,
        gold_standard=gold_standard,
        model_name=model_name,
        model_string=model_string,
        k_values=K_VALUES
    )
    duration = round(time.time() - start, 1)
    print(f"  Time taken: {duration}s")
    all_results.extend(results)

print("\n✓ All models evaluated.")


Evaluating: MiniLM (baseline)
Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 799.18it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for MiniLM (baseline):
  Hit Rate @1: 87% (13/15)
  Hit Rate @3: 93% (14/15)
  Hit Rate @5: 93% (14/15)
  Time taken: 21.1s

Evaluating: BGE-small
Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1148.96it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for BGE-small:
  Hit Rate @1: 87% (13/15)
  Hit Rate @3: 93% (14/15)
  Hit Rate @5: 93% (14/15)
  Time taken: 22.8s

Evaluating: BGE-large
Loading model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 660.67it/s, Materializing param=pooler.dense.weight]                                
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for BGE-large:
  Hit Rate @1: 93% (14/15)
  Hit Rate @3: 100% (15/15)
  Hit Rate @5: 100% (15/15)
  Time taken: 83.0s

✓ All models evaluated.


# 11 Summary table

In [84]:
df_all = pd.DataFrame(all_results)

summary_rows = []
for model_name in MODELS_TO_EVALUATE.keys():
    model_df = df_all[df_all["model"] == model_name]
    row = {"Model": model_name}
    for k in K_VALUES:
        hits     = model_df[f"hit@{k}"].sum()
        total    = len(model_df)
        hit_rate = hits / total * 100
        row[f"Hit Rate @{k}"] = f"{hit_rate:.0f}% ({hits}/{total})"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("=== SUMMARY: Hit Rate by Model ===")
print(summary_df.to_string(index=False))

=== SUMMARY: Hit Rate by Model ===
            Model Hit Rate @1  Hit Rate @3  Hit Rate @5
MiniLM (baseline) 87% (13/15)  93% (14/15)  93% (14/15)
        BGE-small 87% (13/15)  93% (14/15)  93% (14/15)
        BGE-large 93% (14/15) 100% (15/15) 100% (15/15)


# 12 Where each model failed

In [85]:
print("=== MISSES AT @3 ===")
misses = df_all[df_all["hit@3"] == False][["model", "query", "expected_standard", "retrieved_codes"]]

if len(misses) == 0:
    print("All queries hit at @3 for all models!")
else:
    for _, row in misses.iterrows():
        print(f"\n  Model:    {row['model']}")
        print(f"  Query:    {row['query'][:80]}")
        print(f"  Expected: {row['expected_standard']}")
        print(f"  Got:      {row['retrieved_codes'][:3]}")

=== MISSES AT @3 ===

  Model:    MiniLM (baseline)
  Query:    What instructions and information must be given to patients when they leave the 
  Expected: ACC.4.1
  Got:      ['ACC.4.4', 'ACC.3', 'ACC.4.4']

  Model:    BGE-small
  Query:    What instructions and information must be given to patients when they leave the 
  Expected: ACC.4.1
  Got:      ['ACC.4.4', 'ACC.4.4', 'ACC.4.4']


# 13 Save results

In [87]:
# Save summary
summary_df.to_csv("evaluation_summary.csv", index=False)
print("Saved: evaluation_summary.csv")

# Save detailed results
df_save = df_all.copy()
df_save["retrieved_codes"] = df_save["retrieved_codes"].apply(lambda x: ", ".join(x))
df_save.to_csv("evaluation_detailed.csv", index=False)
print("Saved: evaluation_detailed.csv")


Saved: evaluation_summary.csv
Saved: evaluation_detailed.csv
